---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md) — describe the notebook name, cell number, and what went wrong.
*Search [existing issues](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues) first to avoid duplicates.*

# 🎯 Bayesian Thinking
**Extended · Pattern Recognition for the Rest of Us**

> Classical statistics asks: "What is the probability of this data given my hypothesis?" Bayesian statistics asks the more natural question: "What is the probability of my hypothesis given this data?"

### What you'll learn
- Prior, likelihood, posterior — the three Bayesian ingredients
- Bayes' theorem applied to real problems
- Credible intervals vs confidence intervals
- Updating beliefs with new evidence
- Conjugate priors — closed-form Bayesian updates

### Time: ~50 minutes

In [ ]:
# ⚠️  Run this cell first — sets up data and imports
# Tip: Runtime → Run all  (runs everything top to bottom automatically)

import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.grid': True, 'grid.alpha': 0.4,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11
})

from scipy import stats
from scipy.stats import norm, beta, binom

# ── Load / generate data ──────────────────────────────────────────────────────
# Synthetic data — no CSV needed

print(f'  Python {sys.version.split()[0]} | pandas {pd.__version__}')
print('✓ Setup complete')


## 📐 Part 1 — Bayes' Theorem

```
P(θ | data) = P(data | θ) × P(θ) / P(data)
posterior   =  likelihood  ×  prior  / evidence
```

- **Prior P(θ):** what we believe before seeing data
- **Likelihood P(data|θ):** how probable is the data given parameter θ
- **Posterior P(θ|data):** updated belief after seeing data
- **Evidence P(data):** normalizing constant (often ignored — we care about the shape)

The key insight: Bayesian inference is about **updating beliefs** with evidence. Start with a prior, observe data, get a posterior.

In [ ]:
# Classic example: estimating a coin's fairness
# Prior: we think coin is fair (Beta(10,10) — moderate belief in p=0.5)
# Data: flip 20 times, get 15 heads
# Posterior: updated belief about p (probability of heads)

np.random.seed(42)
prior_a, prior_b = 10, 10    # prior: Beta(10,10) centered at 0.5
heads, flips = 15, 20        # observed data

# Beta-Binomial conjugacy: posterior is Beta(a+heads, b+tails)
post_a = prior_a + heads
post_b = prior_b + (flips - heads)

p_range = np.linspace(0, 1, 300)
prior_dist = beta.pdf(p_range, prior_a, prior_b)
likelihood  = binom.pmf(heads, flips, p_range) * 10  # scaled for visibility
posterior   = beta.pdf(p_range, post_a, post_b)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(p_range, prior_dist, color='#888',    lw=2, ls='--', label=f'Prior: Beta({prior_a},{prior_b}) — "probably fair"')
ax.plot(p_range, likelihood,  color='#e85d20', lw=2, ls=':',  label=f'Likelihood: {heads} heads in {flips} flips (scaled)')
ax.plot(p_range, posterior,   color='#1e5fa8', lw=2.5,        label=f'Posterior: Beta({post_a},{post_b})')
ax.axvline(0.5, color='black', lw=0.8, ls='--', alpha=0.4, label='p=0.5 (fair coin)')
ax.axvline(post_a/(post_a+post_b), color='#1a7a45', lw=1.5, ls='--',
           label=f'Posterior mean = {post_a/(post_a+post_b):.2f}')
ax.set_xlabel('p = P(heads)')
ax.set_ylabel('Density')
ax.set_title('Bayesian Update: Coin Fairness\nPrior → Likelihood → Posterior')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()
ci_lo, ci_hi = beta.ppf([0.025, 0.975], post_a, post_b)
print(f"\nPrior mean:     {prior_a/(prior_a+prior_b):.2f}")
print(f"MLE (freq):     {heads/flips:.2f}  (just heads/total — ignores prior)")
print(f"Posterior mean: {post_a/(post_a+post_b):.2f}  (pulled toward prior)")
print(f"95% Credible interval: [{ci_lo:.3f}, {ci_hi:.3f}]")
print("\n📌 The posterior is pulled toward 0.5 by the prior.")
print("   With more data, the posterior converges to the MLE regardless of prior.")

In [ ]:
# Show how prior strength affects the posterior
priors = [(1,1,'Flat prior (no opinion)'), (2,2,'Weak prior'), (10,10,'Moderate prior'), (50,50,'Strong prior')]
fig, axes = plt.subplots(1,4,figsize=(16,4))

for ax, (a,b,label) in zip(axes, priors):
    post_a2, post_b2 = a+heads, b+(flips-heads)
    ax.plot(p_range, beta.pdf(p_range,a,b),         color='#888',    lw=1.5, ls='--', label='Prior')
    ax.plot(p_range, beta.pdf(p_range,post_a2,post_b2), color='#1e5fa8', lw=2,    label='Posterior')
    ax.axvline(0.5, color='black', lw=0.8, ls=':', alpha=0.4)
    ax.set_title(f'{label}\nPosterior mean={post_a2/(post_a2+post_b2):.2f}', fontsize=9)
    ax.set_xlabel('p'); ax.legend(fontsize=8)

plt.suptitle(f'Effect of Prior Strength — same data ({heads}/{flips} heads)', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()
print("📌 Strong priors pull the posterior further from the data.")
print("   With enough data, all priors eventually yield the same posterior.")

In [ ]:
# Sequential updating — beliefs evolve as new data arrives
np.random.seed(7)
true_p = 0.65   # true coin bias (unknown to us)
n_flips = 50
flips_seq = np.random.binomial(1, true_p, n_flips)

a, b = 1, 1  # start with flat prior

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
checkpoints = [1,2,5,10,15,20,30,40,50]
plot_points = [1,2,5,10,20,50]

for ax, n in zip(axes.flatten(), plot_points):
    heads_so_far = flips_seq[:n].sum()
    post_a2 = a + heads_so_far
    post_b2 = b + (n - heads_so_far)
    post_mean = post_a2 / (post_a2+post_b2)
    ci = beta.ppf([0.025,0.975], post_a2, post_b2)
    ax.plot(p_range, beta.pdf(p_range, post_a2, post_b2), color='#1e5fa8', lw=2)
    ax.axvline(true_p, color='#e85d20', lw=1.5, ls='--', label=f'True p={true_p}')
    ax.fill_between(p_range, beta.pdf(p_range, post_a2, post_b2),
                    where=(p_range >= ci[0]) & (p_range <= ci[1]),
                    alpha=0.2, color='#1e5fa8', label='95% CI')
    ax.set_title(f'After {n} flips\n({heads_so_far} heads)\nMean={post_mean:.2f}', fontsize=9)
    ax.legend(fontsize=7)

plt.suptitle('Sequential Bayesian Updating — Posterior Converges to True p=0.65', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 🔄 Part 2 — Credible Intervals vs Confidence Intervals

These sound similar but mean very different things:

**Frequentist 95% Confidence Interval:** If we repeated the experiment 100 times, 95 of the resulting intervals would contain the true parameter. The parameter is fixed — we make statements about the *procedure*.

**Bayesian 95% Credible Interval:** Given the data we observed, there is a 95% probability that the parameter lies in this interval. The parameter is *random* — we make direct probability statements about it.

Most practitioners want the Bayesian interpretation but use frequentist intervals. Bayesian intervals give you what you actually want.

In [ ]:
# Compare: CI vs Credible Interval
np.random.seed(42)
n_experiments = 50
n_obs = 30
true_mu = 5.0
sigma = 2.0

fig, ax = plt.subplots(figsize=(12, 7))
freq_contains, bayes_contains = 0, 0

for i in range(n_experiments):
    data = np.random.normal(true_mu, sigma, n_obs)
    xbar = data.mean()
    se = sigma / np.sqrt(n_obs)
    
    # Frequentist CI
    ci_lo, ci_hi = xbar - 1.96*se, xbar + 1.96*se
    freq_ok = ci_lo <= true_mu <= ci_hi
    freq_contains += freq_ok
    color_f = '#1e5fa8' if freq_ok else '#e85d20'
    ax.plot([ci_lo, ci_hi], [i-0.15, i-0.15], color=color_f, lw=1.5, alpha=0.7)
    
    # Bayesian credible interval (flat prior → same as CI here, but interpretation differs)
    post_mu  = xbar  # with flat prior, posterior mean = sample mean
    post_std = se
    cr_lo, cr_hi = norm.ppf([0.025,0.975], post_mu, post_std)
    bayes_ok = cr_lo <= true_mu <= cr_hi
    bayes_contains += bayes_ok

ax.axvline(true_mu, color='black', lw=2, label=f'True μ = {true_mu}')
ax.set_xlabel('μ')
ax.set_yticks([])
ax.set_title(f'50 Experiments — Frequentist 95% CI\nBlue = contains true μ | Red = misses | {freq_contains}/50 contain true μ')
ax.legend()
plt.tight_layout()
plt.show()
print("\n📌 Frequentist: the 95% refers to the long-run coverage of the PROCEDURE")
print("   Bayesian: the 95% refers to our BELIEF about this specific interval")

## 💼 Exercise

Using a Beta-Binomial model, estimate the true conversion rate of a website that received 450 visits and 63 conversions. Plot the prior (Beta(1,1)), likelihood, and posterior. Report the 95% credible interval.

In [ ]:
# ── Exercise workspace ──────────────────────────────────────────────────────
# Write your code here



In [ ]:
# @title 📝 Quiz — Bayesian Thinking
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What are the three components of Bayes' theorem?
# @markdown **Q2:** What is a prior distribution and where does it come from?
# @markdown **Q3:** What is the key difference between a 95% CI and a 95% credible interval?
# @markdown **Q4:** Why does a strong prior have less influence when n is large?
# @markdown **Q5:** Name one practical situation where a Bayesian approach is more natural than frequentist

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

### 📤 Submit your results

In [ ]:
_NB_NAME_="Bayesian Thinking"
# @title 📋 Quiz Submission
# @markdown **Click ▶ Run** → copy the output box (click the copy icon in the top-left of the output box, or click ⋮ → Copy) → paste into Gemini in this Colab session.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(
        f"Q{i}: {str(v).strip() or '(no answer)'}"
        for i, (_, v) in enumerate(answers.items(), 1)
    )
    print(f"Please grade my quiz answers for the \"{_NB_TITLE}\" notebook")
    print(f"from \"Data Pattern Recognition for the Rest of Us\" (based on ISLP).")
    if GITHUB_USERNAME.strip():
        print(f"Student: @{GITHUB_USERNAME.strip()}")
    print()
    print(qa)
    print()
    print("For each question:")
    print("  1. Say CORRECT, PARTIAL, or INCORRECT")
    print("  2. Explain in 2-3 sentences why — what did I get right or wrong")
    print("  3. Give the ideal complete answer so I know exactly what was expected")
    print("  4. If I was wrong or partial, tell me the specific concept to review")
    print("     and where it appears in the notebook (e.g. Part 2, the SHAP beeswarm plot)")
    print()
    print("End with:")
    print("  - Overall grade: Excellent / Good / Needs Review / Incomplete")
    print("  - A short study plan: which questions to re-read and what to focus on")
    print()
    print("After grading, I will paste specific outputs, charts, or code blocks")
    print("from the notebook — please explain them in detail and answer follow-up questions.")


---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md) — describe the notebook name, cell number, and what went wrong. This is how real open-source projects work.
*Search [existing issues](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues) first to avoid duplicates.*

---
*Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*